# 03 Regime Detection

This notebook fits the unsupervised regime model, inspects transition behavior, and overlays the inferred state labels on price. The default configuration uses a Hidden Markov Model, with GMM available as a baseline.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.backtester import WalkForwardBacktester
from src.regime_detection import RegimeDetector
from src.utils import load_yaml_config

config = load_yaml_config(PROJECT_ROOT / "config" / "parameters.yaml")
processed_dir = PROJECT_ROOT / config["data"]["processed_data_dir"]
figures_dir = PROJECT_ROOT / config["reporting"]["figures_dir"]
figures_dir.mkdir(parents=True, exist_ok=True)

features = pd.read_csv(processed_dir / "nsei_features.csv", index_col=0, parse_dates=True)
detector = RegimeDetector(
    method=config["regime_detection"]["method"],
    n_regimes=config["regime_detection"]["n_regimes"],
    covariance_type=config["regime_detection"]["covariance_type"],
    n_iter=config["regime_detection"]["n_iter"],
    random_state=config["project"]["random_seed"],
)
result = detector.fit_predict(features, feature_columns=config["regime_detection"]["feature_columns"])
regime_frame = features.join(result.assignments)
regime_frame.to_csv(processed_dir / "nsei_regimes.csv")


In [ ]:
helper = WalkForwardBacktester(model_name="random_forest")
helper.plot_regime_overlay(
    regime_frame,
    price_column="close",
    regime_column="regime_id",
    save_path=figures_dir / "regime_overlay.png",
)
helper.plot_transition_matrix_heatmap(
    result.transition_matrix,
    save_path=figures_dir / "transition_matrix_heatmap.png",
)
result.duration_stats
